In [1]:
user_id = "tp99965073"
domain = "AXISB.COM"
app_name = "Fraud_&_AML"
yarn_queue = "root.DEV.AML_FRAUD_CS"

EXECUTOR_MEMORY=8
EXECUTOR_CORE=4

INITIAL_EXECUTORS=2
MAX_EXECUTORS=4
MIN_EXECUTORS=2

In [2]:
import sys
from pyspark.sql import SparkSession, SQLContext
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark.sql.types as T
from datetime import datetime, date,timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import time
import numpy as np


from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))


# Construct the keytab path and the principal
keytab_path = f"/home/{user_id}@axisb.com/{user_id}.keytab"
principal = f"{user_id}@{domain}"

# Execute the commands
!kdestroy
!kinit -k -t "{keytab_path}" {principal}
try:
    spark.stop()
    print("Existing Spark Session Killed")
except:
    print("")
print("Creating new spark session since: ", datetime.now())
spark = SparkSession\
    .builder\
    .appName(f'JH_{user_id}_{app_name}')\
    .config('spark.sql.execution.arrow.enabled', "true")\
    .config('spark.master','yarn')\
    .config('spark.deploy-mode', 'cluster')\
    .config("spark.principal",f"{user_id}@AXISB.COM") \
    .config("spark.keytab",f"{user_id}.keytab")\
    .config('spark.dynamicAllocation.enabled','true') \
    .config('spark.dynamicAllocation.minExecutors',MIN_EXECUTORS) \
    .config('spark.dynamicAllocation.initialExecutors',INITIAL_EXECUTORS) \
    .config('spark.dynamicAllocation.maxExecutors',MAX_EXECUTORS) \
    .config('spark.executor.cores',EXECUTOR_CORE) \
    .config('spark.dynamicAllocation.executorAllocationRatio','0.8')\
    .config('spark.executor.memory',f'{EXECUTOR_MEMORY}g')\
    .config('spark.driver.memory','1g')\
    .config('spark.driver.cores','1')\
    .config("spark.sql.broadcastTimeout", "-1")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")\
    .config("spark.yarn.queue", yarn_queue)\
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')\
    .config('spark.hadoop.hive.exec.dynamic.partition.mode', 'nonstrict')\
    .config('spark.sql.parquet.output.committer.class', 'org.apache.parquet.hadoop.ParquetOutputCommitter')\
    .config('spark.sql.sources.commitProtocolClass', 'org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol')\
    .config("spark.jars", "hdfs:///jars/iceberg-spark-runtime-3.3_2.12-1.6.1.jar,/opt/cloudera/parcels/CDH/jars/ojdbc8.jar")\
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')\
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')\
    .config('spark.sql.catalog.spark_catalog.type', 'hive')\
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.adaptive.coalescePartition.enabled', 'true')\
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')\
    .config('spark.sql.adaptive.autoBroadcastJoin.Threshold','50MB')\
    .config("spark.sql.legacy.allowNonEmptyLocationInCTAS", "true")\
    .enableHiveSupport()\
    .getOrCreate()
sc = SparkContext.getOrCreate()
print("New Spark Session created at : ", datetime.now())

new_log_level = "ERROR"
sc.setLogLevel(new_log_level)
print(f"Logs level set to {new_log_level}")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


Creating new spark session since:  2026-06-19 01:26:40.793092


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/06/19 01:26:43 WARN  util.Utils: [Thread-9]: Ser

New Spark Session created at :  2026-06-19 01:26:54.346159
Logs level set to ERROR


# 01 - PL Graph Base Cohort: LDDF + LADF + LAPMF + SLAD

Objective: convert the accepted PL MoM/dashboard logic into one graph-ready account-level base cohort.

## Inputs and Output

Inputs:

| Short Name | Table | Grain | Use |
|---|---|---|---|
| LDDF | `biu_dm.loan_disb_daily_fact` | disbursal/account | PL base, amount, disbursal month |
| LADF | `biu_dm.loan_application_daily_fact` | application | application/login/sanction metadata |
| LAPMF | `biu_dm.LOAN_ACCOUNT_PERF_MONTHLY_FACT_APPEND` | account-month | FPD/SPD, DPD, ever flags |
| SLAD | `biu_dm.SUPER_LOAN_ACCOUNT_DIM` | account/label | SF, IF, HC09 labels |

Output:

`payment_cards_dev.TP99965073_pl_graph_base_cohort_v1`

Validation targets:

- historical clean LDDF PL base was `8,140,852`; latest observed run was `8,158,743`;
- SLAD match near `100%`;
- LAPMF match near `99.36%`;
- Dec 2025 current data near `68,888` accounts vs Aryan dashboard `68,841`;
- IF comes from `final_flag = 'IF'`; do not search for a separate `if_flag`.
- do not use `LDDF.acct_cls_date` in the raw PL base query because it can contain out-of-range timestamps and break Parquet reads.
- notebook 01 builds the raw PL base: all clean PL disbursed accounts, not open-account-only. Notebook 02b applies the SA-customer scope gate for the graph / algorithm cohort.

## Time Windows Used

This notebook has three different date concepts:

| Purpose | Window / Logic |
|---|---|
| Raw PL base | All available clean LDDF Personal Loan disbursals, after null removal and deduplication |
| Graph / algorithm cohort | Applied in notebook 02b: SA-scoped PL accounts after `LDDF.cust_id_finacle -> SA_ACCOUNT_DIM` validation |
| Open-account scope | Not applied here; no `acct_cls_date` or open-account filter |
| Disbursal month | `TRUNC(last_disbursal_dt, 'MM')` |
| Dashboard sanity | Jun-Dec 2025: `2025-06-01 <= disbursal_month < 2026-01-01` |
| Current DPD snapshot | Parameterized `CURRENT_REP_MONTH`; mentor SQL example uses `202603` |
| LADF onboarding metadata coverage | Track with flags because validated LADF PL coverage starts from 2023-04-01 |

Do not restrict the raw PL base to Jun-Dec 2025. Jun-Dec is only for dashboard reconciliation.
Do not call this raw base "open accounts only". If an open-only view is required, validate a safe `account_status` rule separately and re-baseline counts.
Do not call this raw base the final graph cohort. The graph cohort is SA-scoped in notebook 02b.

In [15]:
# Parameters
EMP_ID = "TP99965073"
OUTPUT_DB = "payment_cards_dev"  # ONLY writable schema (write access granted). All source schemas are READ-ONLY.
WRITE_OUTPUT = True   # Set True only after reviewing validation output.

LDDF_TABLE = "biu_dm.loan_disb_daily_fact"
LADF_TABLE = "biu_dm.loan_application_daily_fact"
LAPMF_TABLE = "biu_dm.LOAN_ACCOUNT_PERF_MONTHLY_FACT_APPEND"
LAPMF_HIST_FALLBACK_TABLE = "biu_dm.LOAN_ACCOUNT_PERF_MONTHLY_FACT_HIST_APPEND"
SLAD_TABLE = "biu_dm.SUPER_LOAN_ACCOUNT_DIM"

OUT_BASE = f"{OUTPUT_DB}.{EMP_ID}_pl_graph_base_cohort_v1"
OUT_QC = f"{OUTPUT_DB}.{EMP_ID}_pl_base_cohort_qc_v1"

DASHBOARD_START = "2025-06-01"
DASHBOARD_END = "2026-01-01"
# Latest "current state" snapshot -- feeds ONLY the investigation/early-warning columns (current_dpd,
# current_outbal2, os_*). This is the mentor's own SQL value (WHERE rep_month = 202603) and is intentionally
# the LATEST rep_month, NOT the graph anchor. Sep-Nov 2025 anchoring + trailing-3-month windows live in 02b.
# GUARDRAIL: do NOT feed current_* as point-in-time features to a Sep-Nov anchor (future leakage); for that use
# the month_diff vintage flags (matured_*/dpd_*) and the 02b transaction window instead.
CURRENT_REP_MONTH = 202603
HISTORICAL_CLEAN_BASE = 8140852
LATEST_OBSERVED_CLEAN_BASE = 8158743
EXPECTED_LAPMF_MATCH_RATE = 0.9936
EXPECTED_SLAD_MATCH_RATE = 1.0000
POPULATION_SCOPE = "RAW_ALL_CLEAN_PL_DISBURSED_ACCOUNTS_NO_ACCT_CLS_DATE_FILTER"

In [4]:
# Helper functions
def table_exists(table_name):
    try:
        spark.table(table_name).limit(1).count()
        return True
    except Exception as exc:
        print(f"Table not available or unreadable: {table_name} :: {exc}")
        return False


def col_or_null(df, name, alias=None, dtype="string"):
    alias = alias or name
    if name in df.columns:
        return F.col(name).cast(dtype).alias(alias)
    return F.lit(None).cast(dtype).alias(alias)


def positive_flag(df, col_name):
    if col_name not in df.columns:
        return F.lit(0)
    return F.when(F.upper(F.trim(F.col(col_name).cast("string"))).isin("Y", "1", "YES", "TRUE"), 1).otherwise(0)


def clean_text(col):
    return F.upper(F.trim(col.cast("string")))


def pct(num, den):
    return None if den in (0, None) else float(num) / float(den)


def show_count(label, df):
    value = df.count()
    print(f"{label}: {value:,}")
    return value


def _hdfs_delete_if_orphaned(table_fqn):
    """Delete the managed-table HDFS dir if a prior failed/partial run left it behind, so a rerun can
    overwrite cleanly (avoids 'location already exists'). Only ever touches the given project table dir."""
    db, tbl = table_fqn.split(".")
    base = None
    for r in spark.sql("DESCRIBE DATABASE {}".format(db)).collect():
        vals = [str(v) for v in r]
        if any(v.strip().lower() == "location" for v in vals):
            for v in vals:
                if v.startswith("hdfs") or v.startswith("file:") or v.startswith("/"):
                    base = v
    if not base:
        return
    path = base.rstrip("/") + "/" + tbl.split(".")[-1].lower()
    jvm = spark.sparkContext._jvm; hconf = spark.sparkContext._jsc.hadoopConfiguration()
    P = jvm.org.apache.hadoop.fs.Path(path); fs = P.getFileSystem(hconf)
    if fs.exists(P):
        fs.delete(P, True); print("removed orphaned location:", path)

def save_overwrite(df, table_fqn):
    """Rerun-safe overwrite of ONE project table (drop + delete orphaned dir + saveAsTable)."""
    spark.sql("DROP TABLE IF EXISTS {}".format(table_fqn))
    try:
        _hdfs_delete_if_orphaned(table_fqn)
    except Exception as e:
        print("location cleanup skipped for {}: {}".format(table_fqn, e))
    df.write.mode("overwrite").saveAsTable(table_fqn)


In [5]:
# Refresh LADF because it previously produced a stale metadata/parquet issue.
try:
    spark.sql(f"REFRESH TABLE {LADF_TABLE}")
    print(f"Refreshed {LADF_TABLE}")
except Exception as exc:
    print(f"Refresh skipped or failed for {LADF_TABLE}: {exc}")

Hive Session ID = 7812ba43-059b-43f5-838d-de0fc891ea0c


Refreshed biu_dm.loan_application_daily_fact


In [6]:
# 1. LDDF Personal Loan base: filter, quality check, deduplicate.
lddf_src = spark.table(LDDF_TABLE)

lddf_pl_raw = (
    lddf_src
    .where(clean_text(F.col("product")) == F.lit("PERSONAL LOAN"))
    .select(
        F.trim(F.col("agreement_no").cast("string")).alias("agreement_no"),
        F.trim(F.col("agreement_id").cast("string")).alias("agreement_id"),
        (F.upper(F.trim(F.col("cust_id_finacle").cast("string"))) if "cust_id_finacle" in lddf_src.columns else F.lit(None).cast("string")).alias("lddf_customer_id"),
        col_or_null(lddf_src, "application_id", "lddf_application_id"),
        F.col("product").alias("lddf_product"),
        F.col("disb_amt_as_on").cast("double").alias("disb_amt_as_on"),
        F.to_date(F.col("last_disbursal_dt")).alias("last_disbursal_dt"),
        col_or_null(lddf_src, "disbursal_month_id", dtype="string"),
        col_or_null(lddf_src, "acct_open_dt", dtype="date"),
        col_or_null(lddf_src, "account_status"),
        col_or_null(lddf_src, "source_system"),
        col_or_null(lddf_src, "rebook_flag"),
    )
)

lddf_quality = spark.createDataFrame([
    ("lddf_pl_raw_rows", lddf_pl_raw.count()),
    ("lddf_pl_distinct_accounts", lddf_pl_raw.select("agreement_no").distinct().count()),
    ("lddf_null_agreement_no", lddf_pl_raw.where(F.col("agreement_no").isNull() | (F.col("agreement_no") == "")).count()),
    ("lddf_null_last_disbursal_dt", lddf_pl_raw.where(F.col("last_disbursal_dt").isNull()).count()),
    ("lddf_null_disb_amt_as_on", lddf_pl_raw.where(F.col("disb_amt_as_on").isNull()).count()),
    ("lddf_non_positive_disb_amt", lddf_pl_raw.where(F.col("disb_amt_as_on") <= 0).count()),
], ["metric", "value"])
lddf_quality.show(truncate=False)
print(f"Population scope: {POPULATION_SCOPE}")
print("No acct_cls_date/open-account filter is applied in this raw PL base.")
print("Graph / algorithm cohort is applied later in notebook 02b after LDDF customer -> SA_ACCOUNT_DIM validation.")

print("LDDF account_status distribution for optional open-only validation:")
(
    lddf_pl_raw
    .groupBy(F.coalesce(F.col("account_status").cast("string"), F.lit("NULL")).alias("account_status"))
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)

lddf_valid = (
    lddf_pl_raw
    .where(F.col("agreement_no").isNotNull() & (F.col("agreement_no") != ""))
    .where(F.col("last_disbursal_dt").isNotNull())
    .where(F.col("disb_amt_as_on").isNotNull())
)

lddf_w = W.partitionBy("agreement_no").orderBy(
    F.col("last_disbursal_dt").desc(),
    F.col("disb_amt_as_on").desc(),
    F.col("agreement_id").desc()
)

lddf_base = (
    lddf_valid
    .withColumn("row_num", F.row_number().over(lddf_w))
    .where(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("disbursal_month", F.trunc(F.col("last_disbursal_dt"), "MM"))
    .withColumn("month_period", F.date_format(F.col("disbursal_month"), "yyyyMM"))
    .withColumn("disb_amt_cr", F.col("disb_amt_as_on") / F.lit(10000000.0))
)

clean_base_count = show_count("Clean LDDF PL base", lddf_base)
print(f"Historical guide snapshot: {HISTORICAL_CLEAN_BASE:,}")
print(f"Latest observed run count: {LATEST_OBSERVED_CLEAN_BASE:,}")

+---------------------------+-------+
|metric                     |value  |
+---------------------------+-------+
|lddf_pl_raw_rows           |8190114|
|lddf_pl_distinct_accounts  |8190113|
|lddf_null_agreement_no     |0      |
|lddf_null_last_disbursal_dt|12     |
|lddf_null_disb_amt_as_on   |12     |
|lddf_non_positive_disb_amt |0      |
+---------------------------+-------+

Population scope: RAW_ALL_CLEAN_PL_DISBURSED_ACCOUNTS_NO_ACCT_CLS_DATE_FILTER
No acct_cls_date/open-account filter is applied in this raw PL base.
Graph / algorithm cohort is applied later in notebook 02b after LDDF customer -> SA_ACCOUNT_DIM validation.
LDDF account_status distribution for optional open-only validation:


+--------------+-------+
|account_status|count  |
+--------------+-------+
|C             |5518632|
|A             |2671482|
+--------------+-------+



[Stage 29:==============================================>         (14 + 3) / 17]

Clean LDDF PL base: 8,190,101
Historical guide snapshot: 8,140,852
Latest observed run count: 8,158,743


In [7]:
# 2. LADF application metadata: collapse to one row per application_id before joining.
ladf_src = spark.table(LADF_TABLE)

ladf_raw = (
    ladf_src
    .where(F.col("application_id").isNotNull())
    .select(
        F.trim(F.col("application_id").cast("string")).alias("ladf_application_id"),
        F.to_date(col_or_null(ladf_src, "logindate", dtype="string")).alias("login_date"),
        col_or_null(ladf_src, "product", "ladf_product"),
        col_or_null(ladf_src, "final_product", "ladf_final_product"),
        F.to_date(col_or_null(ladf_src, "sanction_date", dtype="string")).alias("sanction_date"),
        col_or_null(ladf_src, "login_month_id", dtype="string"),
        col_or_null(ladf_src, "sanction_month_id", dtype="string"),
    )
)

ladf_w = W.partitionBy("ladf_application_id").orderBy(F.col("login_date").desc(), F.col("sanction_date").desc())

ladf_app = (
    ladf_raw
    .withColumn("rn", F.row_number().over(ladf_w))
    .where(F.col("rn") == 1)
    .drop("rn")
    .withColumn("ladf_pl_coverage_flag", F.when(F.col("login_date") >= F.lit("2023-04-01").cast("date"), 1).otherwise(0))
)
show_count("LADF distinct applications", ladf_app)

[Stage 35:====================================================>   (16 + 1) / 17]

LADF distinct applications: 39,443,529


39443529

In [8]:
# 3. LAPMF monthly performance: aggregate to account level BEFORE joining.
lapmf_src = spark.table(LAPMF_TABLE)
if "account_number" not in lapmf_src.columns:
    raise ValueError(f"Expected LAPMF account_number column not found. Columns: {lapmf_src.columns}")

if "final_dpd" in lapmf_src.columns and "dpd" in lapmf_src.columns:
    dpd_expr = F.coalesce(F.col("final_dpd"), F.col("dpd")).cast("double")
elif "final_dpd" in lapmf_src.columns:
    dpd_expr = F.col("final_dpd").cast("double")
elif "dpd" in lapmf_src.columns:
    dpd_expr = F.col("dpd").cast("double")
else:
    dpd_expr = F.lit(None).cast("double")

lapmf_prepped = (
    lapmf_src
    .select(
        F.trim(F.col("account_number").cast("string")).alias("lapmf_account_number"),
        col_or_null(lapmf_src, "mnth", dtype="string"),
        col_or_null(lapmf_src, "rep_month", dtype="int"),
        col_or_null(lapmf_src, "account_open_month", dtype="string"),
        col_or_null(lapmf_src, "month_diff", dtype="int"),
        dpd_expr.alias("dpd_final"),
        positive_flag(lapmf_src, "fpd_flag1").alias("fpd_ind"),
        positive_flag(lapmf_src, "spd_flag1").alias("spd_ind"),
        positive_flag(lapmf_src, "ever30").alias("ever30_ind"),
        positive_flag(lapmf_src, "ever60").alias("ever60_ind"),
        positive_flag(lapmf_src, "ever90").alias("ever90_ind"),
        col_or_null(lapmf_src, "loan_balance", dtype="double"),
        col_or_null(lapmf_src, "emi_amt", dtype="double"),
        col_or_null(lapmf_src, "outbal2", dtype="double"),
        col_or_null(lapmf_src, "os_0_179", dtype="double"),
        col_or_null(lapmf_src, "os_30_179", dtype="double"),
        col_or_null(lapmf_src, "os_60_179", dtype="double"),
        col_or_null(lapmf_src, "os_90_179", dtype="double"),
    )
    .where(F.col("lapmf_account_number").isNotNull() & (F.col("lapmf_account_number") != ""))
)

active_perf_row = (F.col("outbal2") > F.lit(1)) if "outbal2" in lapmf_src.columns else F.lit(True)
current_snapshot_row = active_perf_row & (F.col("rep_month") == F.lit(CURRENT_REP_MONTH))

# Matured-denominator + DPD-vintage flags MATCH the mentor's validated SQL exactly:
#   2m/3m/4m/12m accounts = month_diff >= X ; 6m_accounts = month_diff == 6 (his query uses = 6);
#   3m30 = md==3 & dpd>=30, 4m90 = md==4 & dpd>=90, 6m30 = md==6 & dpd>=30, 12m90 = md==12 & dpd>=90;
#   all gated on outbal2 > 1 (active performance row).
lapmf_acct = (
    lapmf_prepped
    .groupBy("lapmf_account_number")
    .agg(
        F.max("fpd_ind").alias("fpd_flag_final"),
        F.max("spd_ind").alias("spd_flag_final"),
        F.max(F.when((F.col("fpd_ind") == 1) & (F.col("spd_ind") == 1), 1).otherwise(0)).alias("fpd_spd_both_flag"),
        F.max("ever30_ind").alias("ever30_flag"),
        F.max("ever60_ind").alias("ever60_flag"),
        F.max("ever90_ind").alias("ever90_flag"),
        F.max(F.when(F.col("month_diff") == 1, F.col("dpd_final"))).alias("first_cycle_dpd"),
        F.max(F.when(F.col("month_diff") == 2, F.col("dpd_final"))).alias("second_cycle_dpd"),
        F.max("dpd_final").alias("max_dpd_observed"),
        F.max("month_diff").alias("max_mob_observed"),
        F.max(F.when(active_perf_row & (F.col("month_diff") >= 2), 1).otherwise(0)).alias("matured_2m_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") >= 3), 1).otherwise(0)).alias("matured_3m_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") >= 4), 1).otherwise(0)).alias("matured_4m_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") == 6), 1).otherwise(0)).alias("matured_6m_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") >= 12), 1).otherwise(0)).alias("matured_12m_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") == 3) & (F.col("dpd_final") >= 30), 1).otherwise(0)).alias("dpd_3m30_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") == 4) & (F.col("dpd_final") >= 90), 1).otherwise(0)).alias("dpd_4m90_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") == 6) & (F.col("dpd_final") >= 30), 1).otherwise(0)).alias("dpd_6m30_flag"),
        F.max(F.when(active_perf_row & (F.col("month_diff") == 12) & (F.col("dpd_final") >= 90), 1).otherwise(0)).alias("dpd_12m90_flag"),
        F.max(F.when(current_snapshot_row, F.col("dpd_final"))).alias("current_dpd"),
        F.max(F.when(current_snapshot_row, F.col("os_0_179"))).alias("os_0_179"),
        F.max(F.when(current_snapshot_row, F.col("os_30_179"))).alias("os_30_179"),
        F.max(F.when(current_snapshot_row, F.col("os_60_179"))).alias("os_60_179"),
        F.max(F.when(current_snapshot_row, F.col("os_90_179"))).alias("os_90_179"),
        F.max(F.when(current_snapshot_row, F.col("outbal2"))).alias("current_outbal2"),
        F.max("loan_balance").alias("max_loan_balance"),
        F.max("emi_amt").alias("max_emi_amt"),
        F.count("*").alias("lapmf_month_rows"),
    )
)
show_count("LAPMF account-level rows", lapmf_acct)
print(f"LAPMF history fallback available: {table_exists(LAPMF_HIST_FALLBACK_TABLE)}")

LAPMF account-level rows: 32,256,458


[Stage 47:===================================================>(1029 + 7) / 1036]

LAPMF history fallback available: True


In [9]:
# 4. SLAD labels: collapse to one row per account number.
slad_src = spark.table(SLAD_TABLE)
if "accno" not in slad_src.columns:
    raise ValueError(f"Expected SLAD accno column not found. Columns: {slad_src.columns}")

slad_prepped = (
    slad_src
    .select(
        F.trim(F.col("accno").cast("string")).alias("accno"),
        col_or_null(slad_src, "acid", dtype="string"),
        col_or_null(slad_src, "loan_application_id", dtype="string"),
        col_or_null(slad_src, "product_id", dtype="string"),
        col_or_null(slad_src, "product_group", dtype="string"),
        F.to_date(col_or_null(slad_src, "acct_opn_date", dtype="string")).alias("slad_acct_opn_date"),
        clean_text(col_or_null(slad_src, "final_flag")).alias("final_flag_clean"),
        positive_flag(slad_src, "sf_flag").alias("sf_flag_raw"),
        positive_flag(slad_src, "hc_flag").alias("hc_flag_raw"),
        F.to_date(col_or_null(slad_src, "first_emi_date", dtype="string")).alias("first_emi_date"),
        F.to_date(col_or_null(slad_src, "second_emi_date", dtype="string")).alias("second_emi_date"),
    )
    .where(F.col("accno").isNotNull() & (F.col("accno") != ""))
)

slad_acct = (
    slad_prepped
    .groupBy("accno")
    .agg(
        F.max(F.when((F.col("sf_flag_raw") == 1) | (F.col("final_flag_clean") == "SF"), 1).otherwise(0)).alias("sf_flag_final"),
        F.max(F.when(F.col("final_flag_clean") == "IF", 1).otherwise(0)).alias("if_flag_final"),
        F.max(F.when((F.col("hc_flag_raw") == 1) | (F.col("final_flag_clean") == "HC09"), 1).otherwise(0)).alias("hc09_flag_final"),
        F.concat_ws(",", F.sort_array(F.collect_set("final_flag_clean"))).alias("slad_final_flag_values"),
        F.max("first_emi_date").alias("first_emi_date"),
        F.max("second_emi_date").alias("second_emi_date"),
        F.max("slad_acct_opn_date").alias("slad_acct_opn_date"),
        F.first("acid", ignorenulls=True).alias("acid"),
        F.first("loan_application_id", ignorenulls=True).alias("slad_loan_application_id"),
        F.first("product_id", ignorenulls=True).alias("slad_product_id"),
        F.first("product_group", ignorenulls=True).alias("slad_product_group"),
    )
    .withColumn("final_fraud_flag", F.when((F.col("sf_flag_final") == 1) | (F.col("if_flag_final") == 1) | (F.col("hc09_flag_final") == 1), 1).otherwise(0))
)
show_count("SLAD account-level rows", slad_acct)

[Stage 52:====================================================>   (16 + 1) / 17]

SLAD account-level rows: 33,182,932


33182932

In [10]:
# 5. Join all tables at account grain.
base_joined = (
    lddf_base.alias("d")
    .join(ladf_app.alias("l"), F.col("d.agreement_id") == F.col("l.ladf_application_id"), "left")
    .join(lapmf_acct.alias("p"), F.col("d.agreement_no") == F.col("p.lapmf_account_number"), "left")
    .join(slad_acct.alias("s"), F.col("d.agreement_no") == F.col("s.accno"), "left")
    .withColumn("ladf_matched_flag", F.when(F.col("l.ladf_application_id").isNotNull(), 1).otherwise(0))
    .withColumn("lapmf_matched_flag", F.when(F.col("p.lapmf_account_number").isNotNull(), 1).otherwise(0))
    .withColumn("slad_matched_flag", F.when(F.col("s.accno").isNotNull(), 1).otherwise(0))
    .withColumn("sf_flag_final", F.coalesce(F.col("s.sf_flag_final"), F.lit(0)))
    .withColumn("if_flag_final", F.coalesce(F.col("s.if_flag_final"), F.lit(0)))
    .withColumn("hc09_flag_final", F.coalesce(F.col("s.hc09_flag_final"), F.lit(0)))
    .withColumn("final_fraud_flag", F.coalesce(F.col("s.final_fraud_flag"), F.lit(0)))
    .withColumn("fpd_flag_final", F.coalesce(F.col("p.fpd_flag_final"), F.lit(0)))
    .withColumn("spd_flag_final", F.coalesce(F.col("p.spd_flag_final"), F.lit(0)))
    .withColumn("fpd_spd_both_flag", F.coalesce(F.col("p.fpd_spd_both_flag"), F.lit(0)))
    .withColumn("ticket_size_bucket", F.when(F.col("disb_amt_as_on") < 100000, "<1L")
        .when(F.col("disb_amt_as_on") < 300000, "1L-3L")
        .when(F.col("disb_amt_as_on") < 500000, "3L-5L")
        .when(F.col("disb_amt_as_on") < 1000000, "5L-10L")
        .otherwise("10L+"))
    .withColumn("application_metadata_missing_flag", F.when(F.col("ladf_matched_flag") == 0, 1).otherwise(0))
    .withColumn("outside_ladf_pl_coverage_flag", F.when(F.col("last_disbursal_dt") < F.lit("2023-04-01").cast("date"), 1).otherwise(0))
)

base_cohort = base_joined.select(
    "agreement_no", "agreement_id", "lddf_application_id", "ladf_application_id", "accno",
    "lddf_customer_id", "acid", "slad_loan_application_id", "last_disbursal_dt", "disbursal_month", "month_period",
    "acct_open_dt", "slad_acct_opn_date", "login_date", "sanction_date", "first_emi_date", "second_emi_date",
    "lddf_product", "ladf_product", "ladf_final_product", "slad_product_id", "slad_product_group",
    "source_system", "rebook_flag", "account_status", "disb_amt_as_on", "disb_amt_cr", "ticket_size_bucket",
    "fpd_flag_final", "spd_flag_final", "fpd_spd_both_flag", "first_cycle_dpd", "second_cycle_dpd",
    "ever30_flag", "ever60_flag", "ever90_flag", "max_dpd_observed", "max_mob_observed",
    "matured_2m_flag", "matured_3m_flag", "matured_4m_flag", "matured_6m_flag", "matured_12m_flag",
    "dpd_3m30_flag", "dpd_4m90_flag", "dpd_6m30_flag", "dpd_12m90_flag",
    "current_dpd", "os_0_179", "os_30_179", "os_60_179", "os_90_179", "current_outbal2",
    "max_loan_balance", "max_emi_amt", "lapmf_month_rows",
    "sf_flag_final", "if_flag_final", "hc09_flag_final", "final_fraud_flag", "slad_final_flag_values",
    "ladf_matched_flag", "lapmf_matched_flag", "slad_matched_flag", "application_metadata_missing_flag", "outside_ladf_pl_coverage_flag"
)

base_cohort.cache()
show_count("Final base cohort rows", base_cohort)

[Stage 62:=====================================================>(182 + 1) / 183]

Final base cohort rows: 8,190,101


8190101

In [11]:
# 6. Reconciliation and join coverage checks.
base_n = base_cohort.count()
ladf_match_n = base_cohort.where(F.col("ladf_matched_flag") == 1).count()
lapmf_match_n = base_cohort.where(F.col("lapmf_matched_flag") == 1).count()
slad_match_n = base_cohort.where(F.col("slad_matched_flag") == 1).count()

qc_rows = [
    ("base_rows", base_n, None),
    ("distinct_accounts", base_cohort.select("agreement_no").distinct().count(), None),
    ("ladf_matched", ladf_match_n, pct(ladf_match_n, base_n)),
    ("lapmf_matched", lapmf_match_n, pct(lapmf_match_n, base_n)),
    ("slad_matched", slad_match_n, pct(slad_match_n, base_n)),
    ("customer_id_present", base_cohort.where(F.col("lddf_customer_id").isNotNull() & (F.col("lddf_customer_id") != "")).count(), None),
    ("sf_count", base_cohort.where(F.col("sf_flag_final") == 1).count(), None),
    ("final_fraud_count", base_cohort.where(F.col("final_fraud_flag") == 1).count(), None),
    ("fpd_count", base_cohort.where(F.col("fpd_flag_final") == 1).count(), None),
    ("spd_count", base_cohort.where(F.col("spd_flag_final") == 1).count(), None),
    ("dpd_3m30_count", base_cohort.where(F.col("dpd_3m30_flag") == 1).count(), None),
    ("dpd_4m90_count", base_cohort.where(F.col("dpd_4m90_flag") == 1).count(), None),
    ("dpd_6m30_count", base_cohort.where(F.col("dpd_6m30_flag") == 1).count(), None),
    ("dpd_12m90_count", base_cohort.where(F.col("dpd_12m90_flag") == 1).count(), None),
    ("outside_ladf_pl_coverage", base_cohort.where(F.col("outside_ladf_pl_coverage_flag") == 1).count(), None),
]

qc_df = spark.createDataFrame(qc_rows, ["metric", "count_value", "rate_value"])
qc_df.show(100, truncate=False)

print("Historical Dataset_Guide target and latest observed run:")
print(f"- historical clean base snapshot {HISTORICAL_CLEAN_BASE:,}")
print(f"- latest observed clean base {LATEST_OBSERVED_CLEAN_BASE:,}")
print(f"- SLAD match near {EXPECTED_SLAD_MATCH_RATE:.2%}")
print(f"- LAPMF match near {EXPECTED_LAPMF_MATCH_RATE:.2%}")

+------------------------+-----------+------------------+
|metric                  |count_value|rate_value        |
+------------------------+-----------+------------------+
|base_rows               |8190101    |null              |
|distinct_accounts       |8190101    |null              |
|ladf_matched            |3104135    |0.3790105885141099|
|lapmf_matched           |8154203    |0.9956169038696837|
|slad_matched            |8190101    |1.0               |
|customer_id_present     |8190101    |null              |
|sf_count                |17460      |null              |
|final_fraud_count       |19721      |null              |
|fpd_count               |21505      |null              |
|spd_count               |34466      |null              |
|dpd_3m30_count          |13498      |null              |
|dpd_4m90_count          |5063       |null              |
|dpd_6m30_count          |39982      |null              |
|dpd_12m90_count         |65886      |null              |
|outside_ladf_

In [12]:
# 7. Dashboard sanity for Jun-Dec 2025.
# Rates are expressed as PERCENT (x100) to reconcile directly against Aryan's dashboard.
dashboard_sanity = (
    base_cohort
    .where((F.col("disbursal_month") >= F.lit(DASHBOARD_START).cast("date")) & (F.col("disbursal_month") < F.lit(DASHBOARD_END).cast("date")))
    .groupBy("month_period")
    .agg(
        F.countDistinct("agreement_no").alias("DISB_COUNT"),
        F.sum("sf_flag_final").alias("SF_COUNT"),
        F.round(100.0 * F.sum("sf_flag_final") / F.countDistinct("agreement_no"), 4).alias("PER_SF"),
        F.round(F.sum("disb_amt_as_on") / F.lit(10000000.0), 0).alias("DISB_AMT_Crs"),
        F.round(F.sum(F.when(F.col("sf_flag_final") == 1, F.col("disb_amt_as_on")).otherwise(0.0)) / F.lit(10000000.0), 1).alias("SF_AMT_Crs"),
        F.round(100.0 * F.sum(F.when(F.col("sf_flag_final") == 1, F.col("disb_amt_as_on")).otherwise(0.0)) / F.sum("disb_amt_as_on"), 4).alias("PER_SF_AMT"),
        F.sum("final_fraud_flag").alias("FRAUD_COUNT"),
        F.round(100.0 * F.sum("final_fraud_flag") / F.countDistinct("agreement_no"), 4).alias("FRAUD_RATE"),
        F.sum("ladf_matched_flag").alias("APPLICATIONS"),
    )
    .orderBy("month_period")
)

dashboard_sanity.show(20, truncate=False)
print("Rates (PER_SF, PER_SF_AMT, FRAUD_RATE) are in PERCENT to match the dashboard.")
print("Expected Dec 2025 current-data sanity: DISB_COUNT near 68,888; Aryan screenshot was 68,841 and accepted as source movement.")

[Stage 305:=============================================>       (173 + 8) / 200]

+------------+----------+--------+------+------------+----------+----------+-----------+----------+------------+
|month_period|DISB_COUNT|SF_COUNT|PER_SF|DISB_AMT_Crs|SF_AMT_Crs|PER_SF_AMT|FRAUD_COUNT|FRAUD_RATE|APPLICATIONS|
+------------+----------+--------+------+------------+----------+----------+-----------+----------+------------+
|202506      |56893     |81      |0.1424|3531.0      |3.5       |0.0992    |98         |0.1723    |56893       |
|202507      |65551     |76      |0.1159|4096.0      |3.5       |0.0865    |107        |0.1632    |65551       |
|202508      |66170     |80      |0.1209|4126.0      |4.7       |0.1149    |142        |0.2146    |66170       |
|202509      |69192     |97      |0.1402|4299.0      |5.5       |0.129     |134        |0.1937    |69192       |
|202510      |69874     |102     |0.146 |4278.0      |5.1       |0.1199    |115        |0.1646    |69874       |
|202511      |62453     |63      |0.1009|3944.0      |3.3       |0.0848    |69         |0.1105  

In [13]:
# 8. Wrong-join sanity check: agreement_no must NOT map to LADF.application_id.
wrong_join_n = (
    lddf_base.select("agreement_no").alias("d")
    .join(ladf_app.select("ladf_application_id").alias("l"), F.col("d.agreement_no") == F.col("l.ladf_application_id"), "inner")
    .count()
)
print(f"Wrong join LDDF.agreement_no = LADF.application_id matched rows: {wrong_join_n:,}")
print("Expected: zero or effectively zero. Correct join is LDDF.agreement_id = LADF.application_id.")

[Stage 318:===================================================>   (16 + 1) / 17]

Wrong join LDDF.agreement_no = LADF.application_id matched rows: 0
Expected: zero or effectively zero. Correct join is LDDF.agreement_id = LADF.application_id.


In [16]:
# 9. Save output tables after review (rerun-safe overwrite of our project tables only).
if WRITE_OUTPUT:
    print(f"Writing base cohort to {OUT_BASE}")
    save_overwrite(base_cohort, OUT_BASE)
    print(f"Writing QC table to {OUT_QC}")
    save_overwrite(qc_df, OUT_QC)
else:
    print("WRITE_OUTPUT=False. Review QC first, then set WRITE_OUTPUT=True to save:")
    print(f"  {OUT_BASE}")
    print(f"  {OUT_QC}")


Writing base cohort to payment_cards_dev.TP99965073_pl_graph_base_cohort_v1


Writing QC table to payment_cards_dev.TP99965073_pl_base_cohort_qc_v1


## Next Notebook

After this notebook passes reconciliation and saves `payment_cards_dev.TP99965073_pl_graph_base_cohort_v1`, run:

`02_sa_master_customer_bridge.ipynb`

Do not run graph algorithms until the transaction graph fan-in/fan-out UAT passes against the raw transaction tables.